In [11]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [12]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
224,"If this movie was about a fictional character,...",negative
149,1 out of 10.<br /><br />This is the kind of mo...,negative
803,"As a South African, living in South Africa aga...",negative
277,I wish I would have read more reviews and more...,negative
657,"""Deliverance"" is one of the best exploitation ...",positive


In [13]:
# data preprocessing
def lemmitization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """ remove stop words from the text."""
    stops_words = set(stopwords.words("english"))
    text = [word for word in text.split() if word not in stops_words]
    return " ".join(text)

def remove_numbers(text):
    """remove the words for the text"""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def remove_punctuation(text):
    """remove the punctuation from the text"""
    text = re.sub('[%s]' % string.punctuation, '', text)
    text = text.replace(',',"")
    text = re.sub('\s+', ' ', text)
    text = text.strip()
    return text

def removing_url(text):
    """remove the url from the text"""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """normalize the text"""
    try:
        df['review'] = df['review'].apply(lambda x: x.lower())
        df['review'] = df['review'].apply(removing_url)
        df['review'] = df['review'].apply(remove_punctuation)
        df['review'] = df['review'].apply(remove_numbers)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(lemmitization)

        return df

    except Exception as e:
        print(f"Error in normalize_text: {e}")
        raise e

In [14]:
df = normalize_text(df)
df.head()

,review,sentiment
224,movie fictional character movie could stand ju...,negative
149,br br kind movie cant believe wasted hour life...,negative
803,south african living south africa year stay uk...,negative
277,wish would read review opinion movie rented wa...,negative
657,deliverance one best exploitation film come wo...,positive


In [15]:
df['sentiment'].value_counts()

sentiment
negative    272
positive    228
Name: count, dtype: int64

In [16]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})
df.head()

,review,sentiment
224,movie fictional character movie could stand ju...,0
149,br br kind movie cant believe wasted hour life...,0
803,south african living south africa year stay uk...,0
277,wish would read review opinion movie rented wa...,0
657,deliverance one best exploitation film come wo...,1


In [17]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [18]:
vectorizer = CountVectorizer(max_features=100)
x = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [19]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [20]:
import dagshub
import inspect

print(dagshub.__version__)
print(inspect.signature(dagshub.init))

0.7.3
(repo_name: Optional[str] = None, repo_owner: Optional[str] = None, url: Optional[str] = None, root: Optional[str] = None, host: Optional[str] = None, mlflow: bool = True, dvc: bool = False, patch_mlflow: bool = False)


In [21]:
import dagshub
import mlflow

mlflow.set_tracking_uri("https://dagshub.com/nikhidev/MLOps_atlas.mlflow")
dagshub.init(
    repo_name="MLOps_atlas",
    repo_owner="nikhidev",
    mlflow=True
)
mlflow.set_experiment("Logistic Regression Baseline")

2026-10-16 01:44:53,006 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/nikhidev/MLOps_atlas "HTTP/1.1 200 OK"


Initialized MLflow to track repo "nikhidev/MLOps_atlas"

2026-10-16 01:44:53,014 - INFO - Initialized MLflow to track repo "nikhidev/MLOps_atlas"


Repository nikhidev/MLOps_atlas initialized!

2026-10-16 01:44:53,019 - INFO - Repository nikhidev/MLOps_atlas initialized!


<Experiment: artifact_location='mlflow-artifacts:/c682aad33e4f4cb8825a8f2faf458721', creation_time=1792094853954, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1792094853954, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [22]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")



with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)



2026-10-16 01:44:53,416 - INFO - Starting MLflow run...


2026-10-16 01:44:55,767 - INFO - Logging preprocessing parameters...
2026-10-16 01:44:57,126 - INFO - Initializing Logistic Regression model...
2026-10-16 01:44:57,126 - INFO - Fitting the model...
2026-10-16 01:44:57,435 - INFO - Model training complete.
2026-10-16 01:44:57,435 - INFO - Logging model parameters...
2026-10-16 01:44:57,852 - INFO - Making predictions...
2026-10-16 01:44:57,858 - INFO - Calculating evaluation metrics...
2026-10-16 01:44:57,885 - INFO - Logging evaluation metrics...
2026-10-16 01:44:59,457 - INFO - Saving and logging the model...
2026/10/16 01:44:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-10-16 01:45:27,721 - INFO - Model training and logging completed in 31.95 seconds.
2026-10-16 01:45:27,721 - INFO - Accuracy: 0.72
2026-10-16 01:45:27,721 - INFO - Precision: 0.7142857142857143
2026-10-16 01:45:27,721 - INFO - Recall: 0.7142857142857143
2026-10-16 01:45:27,721 - INFO - F1 Score: 0.7142857142857143


🏃 View run judicious-elk-144 at: https://dagshub.com/nikhidev/MLOps_atlas.mlflow/#/experiments/0/runs/4f4b44df10664668acbc975023d6a850
🧪 View experiment at: https://dagshub.com/nikhidev/MLOps_atlas.mlflow/#/experiments/0
